# DC-QAOA Visualization Notebook
This notebook runs the DC-QAOA Max-Cut pipeline step-by-step and visualizes the problem graph, subgraph partitions, and the final optimized solution.

In [2]:
import sys
import ast
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from pathlib import Path
from typing import Optional

from dc_qaoa import solver as _solver_module
from dc_qaoa.graph_loader import load_graph
from dc_qaoa.partitioner import recursive_partition, PartitionNode
from dc_qaoa.solver import qaoa_solve, setup_qpu, USE_PYQUIL, _local_search
from dc_qaoa.merger import merge
from dc_qaoa.scorer import maxcut_score

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['figure.dpi'] = 100

ModuleNotFoundError: No module named 'networkx'

## 1. Helper Functions for Visualization

In [ ]:
def draw_graph(G: nx.Graph, title: str, node_colors: list = None, pos: dict = None):
    """Helper to draw a NetworkX graph nicely."""
    plt.figure(figsize=(10, 8))
    if pos is None:
        pos = nx.spring_layout(G, seed=42)
        
    if node_colors is None:
        node_colors = ['skyblue'] * G.number_of_nodes()
        
    edges = G.edges(data=True)
    weights = [d.get('weight', 1.0) for u, v, d in edges]
    max_weight = max(weights) if weights else 1.0
    edge_widths = [1 + 3 * (w / max_weight) for w in weights]

    nx.draw(G, pos, 
            node_color=node_colors, 
            with_labels=True, 
            node_size=600, 
            font_size=10, 
            font_color='black',
            font_weight='bold',
            edge_color='gray', 
            width=edge_widths, 
            alpha=0.9)
    
    plt.title(title, fontsize=16)
    plt.margins(x=0.1, y=0.1)
    plt.show()
    return pos

## 2. Configuration & Load Graph

In [ ]:
# ── Pipeline settings ────────────────────────────────────────────────────────
GRAPH_PATH = Path("../dataset_A.csv")
MAX_SIZE   = 8
TOP_T      = 10
METHOD     = "separator"   # "separator" (NaiveLGP) | "community"

# ── QVM / QPU connection ─────────────────────────────────────────────────────
import os
os.environ["QCS_SETTINGS_APPLICATIONS_QVM_URL"] = "http://127.0.0.1:5001"

# ── Solver knobs ─────────────────────────────────────────────────────────────
_solver_module.USE_PYQUIL   = True    # False → stub (brute-force / local search)
_solver_module.LAYER_COUNT  = 1       # QAOA depth p
_solver_module.SHOTS        = 1024
_solver_module.SEED         = 42

# ── Quantum computer target ───────────────────────────────────────────────────
QC_NAME = "9q-square-qvm"

if _solver_module.USE_PYQUIL and QC_NAME:
    print(f"Setting up QPU: {QC_NAME}...")
    setup_qpu(QC_NAME)

# ── Load graph ────────────────────────────────────────────────────────────────
print(f"\nLoading graph from {GRAPH_PATH}...")
G = load_graph(GRAPH_PATH)
print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

global_pos = nx.spring_layout(G, seed=42)
draw_graph(G, "Original Problem Graph", pos=global_pos)


## 2b. Mixer Mode Selection

Choose which QAOA mixer to use.  The mixer defines the *driver Hamiltonian*
$B$ that explores the solution space between cost-layer applications.

| Mode | Operator | Initial state | Ring gates / layer | Notes |
|------|----------|---------------|--------------------|-------|
| `"X"`  | $\sum_j X_j$ | $\|+\rangle^n$ | 0 (single-qubit only) | Standard transverse-field; best for noisy hardware |
| `"XX"` | $\sum_{(i,j)\in\text{ring}} X_iX_j$ | $\|+\rangle^n$ | $n$ CNOT pairs | Stronger correlations; more expressive at equal depth |
| `"XY"` | $\sum_{(i,j)\in\text{ring}} \frac{X_iX_j+Y_iY_j}{2}$ | $\|1^k0^k\rangle$, $k=n/2$ | $2n$ CNOT pairs | **Conserves Hamming weight** — start state must have weight $n/2$ |

> **XY caution**: starting from $|+\rangle^n$ with the XY mixer is wrong —
> the mixer cannot move amplitude between Hamming-weight sectors, so each
> sector evolves in isolation and most never reach a good cut partition.
> The solver initialises $k = \lfloor n/2 \rfloor$ qubits to $|1\rangle$ automatically.


In [ ]:
# ── Mixer mode selection ──────────────────────────────────────────────────────
# Set MIXER_MODE to one of: "X", "XX", "XY"
MIXER_MODE = "X"

_MIXER_DESCRIPTIONS = {
    "X":  ("Standard transverse-field",
           "exp(-i β Σ X_j)  — single-qubit RX rotations, lowest gate cost"),
    "XX": ("Graph-coupled XX",
           "exp(-i β Σ_{ring} X_iX_j)  — two-qubit XX on ring, richer correlations"),
    "XY": ("XY conserving mixer",
           "exp(-i β Σ_{ring} (X_iX_j+Y_iY_j)/2)  — preserves Hamming weight"),
}

if MIXER_MODE not in _MIXER_DESCRIPTIONS:
    raise ValueError(f"Unknown MIXER_MODE {MIXER_MODE!r}. Choose from {list(_MIXER_DESCRIPTIONS)}")

_solver_module.MIXER_MODE = MIXER_MODE
name, formula = _MIXER_DESCRIPTIONS[MIXER_MODE]

print(f"{'─'*55}")
print(f"  Mixer mode  : {MIXER_MODE!r}")
print(f"  Name        : {name}")
print(f"  Operator    : {formula}")
print(f"{'─'*55}")


## 3. Graph Partitioning

In [ ]:
print(f"Partitioning graph (method={METHOD}, max_size={MAX_SIZE})...")
partition_tree = recursive_partition(G, max_size=MAX_SIZE, method=METHOD)
leaves = partition_tree.leaves()

print(f"Created {len(leaves)} leaf subgraphs.")

# Assign a distinct color index to each leaf subgraph
node_to_leaf_idx = {}
for i, leaf in enumerate(leaves):
    for node in leaf.graph.nodes():
        if node not in node_to_leaf_idx:
            node_to_leaf_idx[node] = i

# Map indices to unique colors
cmap = plt.cm.get_cmap('tab20', len(leaves))
colors = [cmap(node_to_leaf_idx.get(n, 0)) for n in G.nodes()]

draw_graph(G, f"Partitioned Graph ({len(leaves)} Subgraphs)", node_colors=colors, pos=global_pos)

## 4. Solve Leaf Subgraphs with QAOA

In [ ]:
subgraph_solutions = {}

for i, leaf in enumerate(leaves):
    n_nodes = leaf.graph.number_of_nodes()
    backend = "pyQuil" if _solver_module.USE_PYQUIL else "stub"
    print(f"\n--- Solving Leaf {i + 1}/{len(leaves)} ---")
    print(f"Nodes: {n_nodes}, Edges: {leaf.graph.number_of_edges()} [{backend}]")
    
    # Solve (QAOA or stub)
    solutions = qaoa_solve(leaf.graph, top_t=TOP_T)
    subgraph_solutions[id(leaf)] = solutions
    
    best = maxcut_score(leaf.graph, solutions[0]) if solutions else 0.0
    print(f"-> Found {len(solutions)} solution(s). Best subgraph cut = {best:.4f}")

## 4b. Optimization History — E[Cut] per COBYLA trial

Each line is one COBYLA multi-start run.  The quantity plotted is
$\mathbb{E}[\text{cut}]$, the **average weighted cut** over all measurement
shots at each COBYLA function evaluation — **higher is better**.

The solver stores these traces in `_solver_module.OPTIMIZATION_HISTORY`
(only populated when `USE_PYQUIL = True`).

> **Why not plot "loss"?**  COBYLA minimises $-\mathbb{E}[\text{cut}]$.
> Plotting the raw cut value is more interpretable — you can read off the
> approximate ratio directly.  A second panel shows the loss (negated) for
> completeness.


In [ ]:
import math

histories = _solver_module.OPTIMIZATION_HISTORY   # {subgraph_id: [[cut_per_iter], ...]}

if not histories:
    print("No optimization history found.  Make sure USE_PYQUIL=True and re-run Step 4.")
else:
    num_sg = len(histories)
    cols   = min(3, num_sg)
    rows   = math.ceil(num_sg / cols)

    # ── One figure with 2 rows of panels: E[Cut] on top, Loss on bottom ──────
    fig, all_axes = plt.subplots(
        rows * 2, cols,
        figsize=(5.5 * cols, 3.5 * rows * 2),
        squeeze=False,
    )
    cut_axes  = all_axes[:rows]   # top half  → E[Cut]
    loss_axes = all_axes[rows:]   # bottom half → Loss = -E[Cut]

    cmap = plt.cm.tab10

    for sg_idx, (subgraph_id, trials) in enumerate(histories.items()):
        row, col = divmod(sg_idx, cols)
        ax_cut  = cut_axes[row][col]
        ax_loss = loss_axes[row][col]

        best_trial_idx = max(range(len(trials)), key=lambda t: max(trials[t], default=-1e9))

        for t_idx, trace in enumerate(trials):
            iterations = range(1, len(trace) + 1)
            is_best    = (t_idx == best_trial_idx)
            lw         = 2.2 if is_best else 1.0
            alpha      = 0.95 if is_best else 0.5
            label      = f"Init {t_idx+1}" + (" ★" if is_best else "")
            color      = cmap(t_idx % 10)

            loss_trace = [-v for v in trace]   # COBYLA minimised this

            ax_cut.plot(iterations, trace,      color=color, lw=lw, alpha=alpha, label=label)
            ax_loss.plot(iterations, loss_trace, color=color, lw=lw, alpha=alpha, label=label)

        # Annotate best final E[Cut]
        best_val = max(max(t, default=0) for t in trials)
        ax_cut.axhline(best_val, color="black", lw=1.0, ls="--", alpha=0.4)
        ax_cut.annotate(
            f"best {best_val:.3f}",
            xy=(1, best_val), xycoords=("axes fraction", "data"),
            fontsize=7, ha="right", color="black", alpha=0.7,
        )

        # ── Titles & labels ──
        ax_cut.set_title(f"Subgraph {sg_idx+1}  —  E[Cut]", fontsize=10)
        ax_cut.set_xlabel("COBYLA function evaluation", fontsize=8)
        ax_cut.set_ylabel("E[Cut]  (higher = better)", fontsize=8)
        ax_cut.grid(True, ls="--", alpha=0.4)
        if len(trials) <= 6:
            ax_cut.legend(fontsize=7, loc="lower right")

        ax_loss.set_title(f"Subgraph {sg_idx+1}  —  Loss = −E[Cut]", fontsize=10)
        ax_loss.set_xlabel("COBYLA function evaluation", fontsize=8)
        ax_loss.set_ylabel("Loss  (lower = better)", fontsize=8)
        ax_loss.grid(True, ls="--", alpha=0.4)

    # Hide empty subplots
    for sg_idx in range(num_sg, rows * cols):
        row, col = divmod(sg_idx, cols)
        cut_axes[row][col].set_visible(False)
        loss_axes[row][col].set_visible(False)

    fig.suptitle(
        f"QAOA Optimization History  |  Mixer: {_solver_module.MIXER_MODE!r}  "
        f"|  p={_solver_module.LAYER_COUNT}  |  {_solver_module.NUM_STARTS} starts",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'Subgraph':<12} {'Trials':>7} {'Iters/trial':>12} {'Best E[Cut]':>12}")
    print("─" * 48)
    for sg_idx, (sid, trials) in enumerate(histories.items()):
        best_v   = max(max(t, default=0) for t in trials)
        avg_iter = sum(len(t) for t in trials) / len(trials) if trials else 0
        print(f"{sg_idx+1:<12} {len(trials):>7} {avg_iter:>12.1f} {best_v:>12.4f}")


## 5. Merger & GR Policy

In [ ]:
print("\nMerging subgraphs via GR policy...")
best_assignment = merge(G, partition_tree, subgraph_solutions, top_t=TOP_T)

# Ensure all nodes have an assignment (fallback to +1)
for n in G.nodes():
    if n not in best_assignment:
        best_assignment[n] = 1

pre_polish_score = maxcut_score(G, best_assignment)
print(f"\nPre-polish Max-Cut Score: {pre_polish_score:.4f}")

## 6. Local Search Polish & Final Result

In [ ]:
print("Running fast local search pass to polish border assignments...")
best_assignment = _local_search(G, list(G.nodes()), best_assignment)
final_score = maxcut_score(G, best_assignment)

total_weight = sum(d.get("weight", 1.0) for _, _, d in G.edges(data=True))

print(f"\n{'=' * 50}")
print(f"  FINAL RESULTS")
print(f"{'=' * 50}")
print(f"Pre-Polish Score : {pre_polish_score:.4f}")
print(f"Final Score      : {final_score:.4f} (+{final_score - pre_polish_score:.4f})")
print(f"Total Edge Weight: {total_weight:.4f}")
print(f"Approx Ratio     : {final_score / total_weight:.4f}")
print(f"{'=' * 50}")

## 7. Visualize Final Cut Assignment

In [ ]:
# +1 spin -> Lightgreen, -1 spin -> Tomato
color_map = {1: 'lightgreen', -1: 'tomato'}
final_colors = [color_map[best_assignment[n]] for n in G.nodes()]

draw_graph(G, f"Final Max-Cut (Score: {final_score:.2f})\nGreen: +1, Red: -1", 
           node_colors=final_colors, pos=global_pos)

## 8. QAOA Cost Landscape — p=1 Grid Survey

For $p=1$ the circuit has exactly two parameters: one $\gamma$ (cost angle)
and one $\beta$ (mixer angle).  Sweeping a 2-D grid and evaluating
$\mathbb{E}[\text{cut}](\gamma,\beta)$ lets you see the optimisation
landscape the COBYLA optimizer is navigating and confirm that your best
found parameters sit near a true peak.

### Why the landscape matters

* Reveals whether the landscape is **smooth and unimodal** (easy for COBYLA)
  or **multi-modal** (needs many restarts or a global optimiser).
* Validates that the parametric circuit is wired correctly — a flat landscape
  means the cost layer has no effect.
* For the `"XY"` mixer the landscape has a narrower peak because the mixer
  conserves Hamming weight, so the search space is smaller but the basin is
  steeper.

### How to hook this into the existing solver

The code below re-uses `_build_qaoa_program` and the compiled executable
directly — no extra circuit wiring needed.

```python
import numpy as np
import matplotlib.pyplot as plt

# ── 1. Pick a single leaf to survey ──────────────────────────────────────────
leaf    = leaves[0]                          # first subgraph
nodes   = list(leaf.graph.nodes())
n       = len(nodes)
node_idx = {v: i for i, v in enumerate(nodes)}
edges   = [(node_idx[u], node_idx[v], float(d.get("weight", 1.0)))
           for u, v, d in leaf.graph.edges(data=True)]

# ── 2. Compile the p=1 circuit once ──────────────────────────────────────────
import solver as _solver_module
from solver import _build_qaoa_program

prog       = _build_qaoa_program(n, edges, p_layers=1,
                                  mixer_mode=_solver_module.MIXER_MODE)
executable = _solver_module._QC.compile(
    prog.wrap_in_numshots_loop(_solver_module.SHOTS)
)

# ── 3. Grid sweep ─────────────────────────────────────────────────────────────
GRID = 30                                    # 30×30 = 900 circuit evaluations
gamma_vals = np.linspace(-np.pi, np.pi, GRID)
beta_vals  = np.linspace(-np.pi, np.pi, GRID)
Z          = np.zeros((GRID, GRID))

for i, g in enumerate(gamma_vals):
    for j, b in enumerate(beta_vals):
        result = _solver_module._QC.run(
            executable,
            memory_map={"gammas": [float(g)], "betas": [float(b)]},
        )
        bits = np.array(result.get_register_map().get("ro"))
        total = sum(
            w for shot in bits
              for (u, v, w) in edges
              if shot[u] != shot[v]
        )
        Z[i, j] = total / len(bits)          # E[cut] at this (γ, β)

# ── 4. Plot ───────────────────────────────────────────────────────────────────
G_grid, B_grid = np.meshgrid(gamma_vals, beta_vals, indexing="ij")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour map
cf = axes[0].contourf(G_grid, B_grid, Z, levels=40, cmap="viridis")
plt.colorbar(cf, ax=axes[0], label="E[Cut]")
axes[0].set_xlabel(r"$\gamma$")
axes[0].set_ylabel(r"$\beta$")
axes[0].set_title(
    f"Cost landscape  |  Mixer={_solver_module.MIXER_MODE!r}  p=1\n"
    f"Subgraph: {n} qubits, {len(edges)} edges"
)

# Scatter the COBYLA starting and final points (if history available)
history = _solver_module.OPTIMIZATION_HISTORY.get(id(leaf))
if history:
    # Rerun just to recover start points from the fixed seed
    rng = np.random.default_rng(_solver_module.SEED)
    for trial in range(_solver_module.NUM_STARTS):
        g0, b0 = rng.uniform(-np.pi, np.pi, 2)
        axes[0].scatter(g0, b0, marker="x", color="red",  s=60, zorder=5)
    axes[0].scatter([], [], marker="x", color="red",  label="COBYLA start")
axes[0].legend()

# 3-D surface
ax3d = fig.add_subplot(1, 2, 2, projection="3d")   # replace axes[1] with 3D
# Note: add `from mpl_toolkits.mplot3d import Axes3D` at top of notebook
ax3d.plot_surface(G_grid, B_grid, Z, cmap="viridis", alpha=0.85)
ax3d.set_xlabel(r"$\gamma$")
ax3d.set_ylabel(r"$\beta$")
ax3d.set_zlabel("E[Cut]")
ax3d.set_title("3-D surface")

plt.tight_layout()
plt.show()

peak_idx = np.unravel_index(Z.argmax(), Z.shape)
print(f"Grid peak  γ={gamma_vals[peak_idx[0]]:.3f}  β={beta_vals[peak_idx[1]]:.3f}"
      f"  E[Cut]={Z[peak_idx]:.4f}")
```

### Performance note

A 30×30 grid × 1024 shots = **921,600 circuit executions**.  On QVM this takes
a few minutes; on real QPU it is expensive.  Reduce `GRID` to 15 or `SHOTS`
to 256 for a quick preview.  Alternatively, use the stub backend
(`USE_PYQUIL=False`) with `_brute_force` to get the exact $\mathbb{E}[\text{cut}]$
analytically (no sampling noise) and scan the landscape without a QVM.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── 1. Pick a single leaf to survey ──────────────────────────────────────────
leaf    = leaves[0]                          # first subgraph
nodes   = list(leaf.graph.nodes())
n       = len(nodes)
node_idx = {v: i for i, v in enumerate(nodes)}
edges   = [(node_idx[u], node_idx[v], float(d.get("weight", 1.0)))
           for u, v, d in leaf.graph.edges(data=True)]

# ── 2. Compile the p=1 circuit once ──────────────────────────────────────────
import solver as _solver_module
from solver import _build_qaoa_program

prog       = _build_qaoa_program(n, edges, p_layers=1,
                                  mixer_mode=_solver_module.MIXER_MODE)
executable = _solver_module._QC.compile(
    prog.wrap_in_numshots_loop(_solver_module.SHOTS)
)

# ── 3. Grid sweep ─────────────────────────────────────────────────────────────
GRID = 30                                    # 30×30 = 900 circuit evaluations
gamma_vals = np.linspace(-np.pi, np.pi, GRID)
beta_vals  = np.linspace(-np.pi, np.pi, GRID)
Z          = np.zeros((GRID, GRID))

for i, g in enumerate(gamma_vals):
    for j, b in enumerate(beta_vals):
        result = _solver_module._QC.run(
            executable,
            memory_map={"gammas": [float(g)], "betas": [float(b)]},
        )
        bits = np.array(result.get_register_map().get("ro"))
        total = sum(
            w for shot in bits
              for (u, v, w) in edges
              if shot[u] != shot[v]
        )
        Z[i, j] = total / len(bits)          # E[cut] at this (γ, β)

# ── 4. Plot ───────────────────────────────────────────────────────────────────
G_grid, B_grid = np.meshgrid(gamma_vals, beta_vals, indexing="ij")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour map
cf = axes[0].contourf(G_grid, B_grid, Z, levels=40, cmap="viridis")
plt.colorbar(cf, ax=axes[0], label="E[Cut]")
axes[0].set_xlabel(r"$\gamma$")
axes[0].set_ylabel(r"$\beta$")
axes[0].set_title(
    f"Cost landscape  |  Mixer={_solver_module.MIXER_MODE!r}  p=1\n"
    f"Subgraph: {n} qubits, {len(edges)} edges"
)

# Scatter the COBYLA starting and final points (if history available)
history = _solver_module.OPTIMIZATION_HISTORY.get(id(leaf))
if history:
    # Rerun just to recover start points from the fixed seed
    rng = np.random.default_rng(_solver_module.SEED)
    for trial in range(_solver_module.NUM_STARTS):
        g0, b0 = rng.uniform(-np.pi, np.pi, 2)
        axes[0].scatter(g0, b0, marker="x", color="red",  s=60, zorder=5)
    axes[0].scatter([], [], marker="x", color="red",  label="COBYLA start")
axes[0].legend()

# 3-D surface
ax3d = fig.add_subplot(1, 2, 2, projection="3d")   # replace axes[1] with 3D
# Note: add `from mpl_toolkits.mplot3d import Axes3D` at top of notebook
ax3d.plot_surface(G_grid, B_grid, Z, cmap="viridis", alpha=0.85)
ax3d.set_xlabel(r"$\gamma$")
ax3d.set_ylabel(r"$\beta$")
ax3d.set_zlabel("E[Cut]")
ax3d.set_title("3-D surface")

plt.tight_layout()
plt.show()

peak_idx = np.unravel_index(Z.argmax(), Z.shape)
print(f"Grid peak  γ={gamma_vals[peak_idx[0]]:.3f}  β={beta_vals[peak_idx[1]]:.3f}"
      f"  E[Cut]={Z[peak_idx]:.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.contourf(G_grid, B_grid, Z, levels=40, cmap="viridis")
plt.scatter()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour map
cf = axes[0].contourf(G_grid, B_grid, Z, levels=40, cmap="viridis")
plt.colorbar(cf, ax=axes[0], label="E[Cut]")
axes[0].set_xlabel(r"$\gamma$")
axes[0].set_ylabel(r"$\beta$")
axes[0].set_title(
    f"Cost landscape  |  Mixer={_solver_module.MIXER_MODE!r}  p=1\n"
    f"Subgraph: {n} qubits, {len(edges)} edges"
)

# Scatter the COBYLA starting and final points (if history available)
history = _solver_module.OPTIMIZATION_HISTORY.get(id(leaf))
if history:
    # Rerun just to recover start points from the fixed seed
    rng = np.random.default_rng(_solver_module.SEED)
    for trial in range(_solver_module.NUM_STARTS):
        g0, b0 = rng.uniform(-np.pi, np.pi, 2)
        axes[0].scatter(g0, b0, marker="x", color="red",  s=60, zorder=5)
    axes[0].scatter([], [], marker="x", color="red",  label="COBYLA start")
axes[0].legend()

# 3-D surface
ax3d = fig.add_subplot(1, 2, 2, projection="3d")   # replace axes[1] with 3D
# Note: add `from mpl_toolkits.mplot3d import Axes3D` at top of notebook
ax3d.plot_surface(G_grid, B_grid, Z, cmap="viridis", alpha=0.85)
ax3d.set_xlabel(r"$\gamma$")
ax3d.set_ylabel(r"$\beta$")
ax3d.set_zlabel("E[Cut]")
ax3d.set_title("3-D surface")

plt.tight_layout()
plt.show()